# BatikCraft — Kaggle Asset Pack Builder

Notebook ini memproses dataset batik menjadi kandidat modular, membuat contact sheet dan
`review.csv`, lalu mengekspor `.batikpack` yang dapat langsung dipasang di BatikCraft Studio.

Pipeline bersifat **semi-otomatis**: computer vision membantu crop/segmentasi, tetapi hasil
wajib dikurasi manusia sebelum menjadi pustaka bawaan.

## 1. Siapkan repository

Tambahkan repository sebagai Kaggle Dataset atau clone ke:

```text
/kaggle/working/BatikCraftStudio
```

Dengan internet Kaggle aktif:

```bash
git clone --depth 1 https://github.com/balyaapik/BatikCraftStudio.git /kaggle/working/BatikCraftStudio
```

In [ ]:
from pathlib import Path
import sys
import subprocess

repo_candidates = [
    Path("/kaggle/working/BatikCraftStudio"),
    Path("/kaggle/input/batikcraftstudio"),
    Path.cwd(),
]
REPO_ROOT = next(
    (path for path in repo_candidates if (path / "src" / "batikcraft_studio").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(
        "Repository BatikCraftStudio belum tersedia. "
        "Clone ke /kaggle/working/BatikCraftStudio atau tambahkan sebagai Kaggle Dataset."
    )

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))
print("Repository:", REPO_ROOT)

In [ ]:
required = {
    "cv2": "opencv-python-headless",
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
}
missing = []
for module_name, package_name in required.items():
    try:
        __import__(module_name)
    except ImportError:
        missing.append(package_name)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import pandas as pd
from PIL import Image
from kaggle_asset_pipeline import (
    ExtractionConfig,
    build_curated_pack,
    extract_dataset,
    make_contact_sheets,
    write_review_files,
)
print("Dependencies ready.")

## 2. Konfigurasi dataset dan output

In [ ]:
DATASET_ROOT = Path("/kaggle/input/datasets/hamdanialikhsan/batik-motifs")
WORK_ROOT = Path("/kaggle/working/batikcraft-asset-builder")

config = ExtractionConfig(
    dataset_root=DATASET_ROOT,
    work_root=WORK_ROOT,
    pack_id="batikcraft-default-library-v1",
    pack_name="BatikCraft Default Library",
    pack_version="1.0.0",
    pack_author="Balya Rochmadi",
    pack_description="Asset modular hasil ekstraksi dan kurasi dataset batik.",
    extraction_modes=("full", "components", "grid"),
    max_source_side=1800,
    grid_size=512,
    grid_overlap=0.25,
    max_candidates_per_image=40,
    auto_accept_confidence=0.86,
    master_asset_size=1024,
    thumbnail_size=192,
)

print("Dataset:", config.dataset_root)
print("Work root:", config.work_root)
print("Output pack:", config.output_pack)

## 3. Ekstrak kandidat modular

In [ ]:
candidates = extract_dataset(config)
print(f"Kandidat tersimpan: {len(candidates):,}")
print("Folder candidate:", config.candidate_root)

## 4. Buat antrean kurasi dan contact sheet

In [ ]:
review_df = write_review_files(candidates, config)
display(review_df.head(20))

contact_sheets = sorted(config.contact_sheet_root.glob("*.jpg"))
print("Review CSV:", config.review_csv)
print("Contact sheets:", len(contact_sheets))
if contact_sheets:
    display(Image.open(contact_sheets[0]))

## 5. Kurasi manusia

Edit `review.csv`:

- `keep=1` untuk diterima, `keep=0` untuk ditolak;
- perbaiki `name`;
- pilih `category`: `motif-pokok`, `isen-isen`, `ornamen`, `tekstur`, atau `lainnya`;
- pisahkan tags dengan `|`;
- tambahkan catatan pada `notes`.

Kandidat confidence tinggi hanya diberi saran awal `keep=1`; tetap periksa contact sheet.
Setelah kurasi di spreadsheet, upload kembali CSV final dan ubah path di cell berikut.

In [ ]:
CURATED_REVIEW_CSV = config.review_csv
# Contoh bila CSV hasil kurasi di-upload sebagai Kaggle input:
# CURATED_REVIEW_CSV = Path("/kaggle/input/batik-review-final/review.csv")

curated = pd.read_csv(CURATED_REVIEW_CSV)
accepted = curated[
    curated["keep"].astype(str).str.casefold().isin({"1", "true", "yes", "y", "keep"})
]
print(f"Disetujui: {len(accepted):,} dari {len(curated):,}")
display(accepted.head(20))

## 6. Bangun dan validasi `.batikpack`

In [ ]:
pack_path = build_curated_pack(
    config,
    curated_review_csv=CURATED_REVIEW_CSV,
)
print("Pack selesai:", pack_path)
print("Ukuran:", f"{pack_path.stat().st_size / 1024 / 1024:.2f} MB")

## 7. Install di BatikCraft Studio

Unduh file output:

```text
/kaggle/working/batikcraft-asset-builder/batikcraft-default-library-v1.batikpack
```

Di aplikasi pilih:

```text
Asset → Install Asset Pack…
```

Paket kemudian muncul di **Pustaka Asset** dan dapat dicari, dipreview, serta dimasukkan
ke canvas tanpa menjalankan Kaggle lagi.